# 06 `fpos` Vs `fmiss`: XAI And Target Asymmetry

This notebook confronts the two targets directly. The goal is not only to show that their leading recipes differ, but to show why the thesis treats them as different scientific problems.

**Questions answered here**
- Which features dominate the best `fpos` model?
- How do family-level behaviors differ between the best `fpos` and best `fmiss` lines?
- Where is the strongest evidence that the two targets should not share one final recipe?


In [ ]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

def _find_thesis_root():
    cwd = Path.cwd().resolve()
    direct = [cwd, *cwd.parents]
    nested = [candidate / "thesis" for candidate in direct]
    for candidate in [*direct, *nested]:
        if (
            (candidate / "src" / "qc_thesis" / "__init__.py").exists()
            and (candidate / "README.md").exists()
            and (candidate / "notebooks").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not find thesis root from notebook session")

ROOT = _find_thesis_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from qc_thesis import *

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)
apply_thesis_style()


def show_saved_figure(path, caption=None):
    figure_path = Path(path)
    if not figure_path.is_absolute():
        figure_path = (ROOT / figure_path).resolve()
    if not figure_path.exists():
        display(Markdown(f"_Missing figure: `{figure_path}`_"))
        return
    if caption:
        display(Markdown(caption))
    display(Image(filename=str(figure_path)))

fig_dir, table_dir = notebook_output_dirs("06_fpos_vs_fmiss_xai")
shap_df = load_fpos_shap_importance()
shap_group_df = build_shap_group_summary()
fpos_family = build_family_summary("fpos_waveform_winner", "fpos", "recording_disjoint_main")
fmiss_family = build_family_summary("fmiss_reduced_latent", "fmiss", "recording_disjoint_main")
asymmetry = build_target_asymmetry_table()
fpos_progress = build_benchmark_progress_table("fpos", scope="core")
fmiss_progress = build_benchmark_progress_table("fmiss", scope="core")


## 1. What drives the best `fpos` model?

The SHAP summary is useful because it tells us whether the lead `fpos` stack is a black box or a structured correction model. The answer from earlier analysis was: a strong transferred prior plus waveform/amplitude/ISI corrections.


In [ ]:
display(shap_df.head(20))
display(shap_group_df)
save_table(shap_df, table_dir, "fpos_shap_importance")
save_table(shap_group_df, table_dir, "fpos_shap_group_summary")
fig, _ = plot_fpos_shap_top(shap_df, top_n=15)
save_figure(fig, fig_dir, "fpos_shap_top")
fig


In [ ]:
show_saved_figure(
    fig_dir / "fpos_shap_group_summary.png",
    "The grouped SHAP summary collapses the feature ranking into thesis-level blocks, making it clearer that the best `fpos` model uses a transferred prior plus waveform, amplitude, and ISI corrections rather than one narrow shortcut.",
)


## 2. Family behavior side by side

The family tables below make the asymmetry concrete. The question is not only which model is better overall, but which families reward or punish each target differently.


In [ ]:
display(fpos_family)
display(fmiss_family)
display(asymmetry)
save_table(fpos_family, table_dir, "fpos_family_summary")
save_table(fmiss_family, table_dir, "fmiss_family_summary")
save_table(asymmetry, table_dir, "fpos_vs_fmiss_family_comparison")


In [ ]:
fig, _, _ = plot_group_metric(fpos_family, group_col="study_set", metric="r2", title="Best `fpos` family behavior")
save_figure(fig, fig_dir, "fpos_family_behavior")
fig


In [ ]:
fig, _, _ = plot_group_metric(fmiss_family, group_col="study_set", metric="r2", title="Best `fmiss` family behavior")
save_figure(fig, fig_dir, "fmiss_family_behavior")
fig


In [ ]:
asymmetry["r2_gap_fpos_minus_fmiss"] = asymmetry["fpos_r2"] - asymmetry["fmiss_r2"]
display(asymmetry.sort_values("r2_gap_fpos_minus_fmiss", ascending=False))
fig, _ = plot_target_asymmetry(asymmetry)
save_figure(fig, fig_dir, "target_asymmetry_combined")
fig


In [ ]:
show_saved_figure(
    fig_dir / "fpos_fmiss_family_r2_scatter.png",
    "This family-level scatter is the compact comparison plot for the thesis: each point is one paired family, with lead `fpos` and lead `fmiss` performance on the two axes.",
)


## 3. Benchmark-level asymmetry

This table compresses the whole thesis finding into one summary: `fpos` gained much more from the later target-aware structure than `fmiss` did.


In [ ]:
basic_fpos = fpos_progress.loc[fpos_progress["recipe_id"] == "fpos_hybrid_only"].iloc[0]
final_fpos = fpos_progress.sort_values("r2", ascending=False).iloc[0]
basic_fmiss = fmiss_progress.loc[fmiss_progress["recipe_id"] == "fmiss_hybrid_only"].iloc[0]
final_fmiss = fmiss_progress.sort_values("r2", ascending=False).iloc[0]
target_story = pd.DataFrame([
    {"target": "fpos", "basic_transfer_r2": basic_fpos["r2"], "final_r2": final_fpos["r2"], "gain_r2": final_fpos["r2"] - basic_fpos["r2"]},
    {"target": "fmiss", "basic_transfer_r2": basic_fmiss["r2"], "final_r2": final_fmiss["r2"], "gain_r2": final_fmiss["r2"] - basic_fmiss["r2"]},
])
display(target_story)
save_table(target_story, table_dir, "target_progress_summary")


In [ ]:
display(Markdown(
    """
## Key takeaways

- The best `fpos` model is heavily shaped by transferred prior information plus waveform/amplitude corrections.
- The best `fmiss` model is more conservative and context-first.
- Family-level asymmetry is not noise; it is evidence that `fpos` and `fmiss` respond to different kinds of transfer signal.
- This notebook is the strongest support for using separate deployment models for the two targets.
"""
))
